## Final Training

Based on the eperiments we have conducted,
for the final model we employ the following regualrization techniques <br>
1. Random rotations 
2. L2 Regularization with $\lambda = 0.005$
3. lowering the learning rate
4. smaller model

In [ ]:
import sys
from google.colab import drive
drive.mount("/content/drive")
PROJECT_ROOT = "/content/drive/MyDrive/Projects/MMSEN"
sys.path.insert(0, PROJECT_ROOT)

In [ ]:
!kaggle datasets download -d andrewmvd/metastatic-tissue-classification-patchcamelyon
!unzip metastatic-tissue-classification-patchcamelyon.zip

In [ ]:
import torch
import random
from torchvision import transforms, models

from src.train import train_loop_regularization
from src.models import MMSEN_small
from src.evaluation import compare_exps, eval

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# define a transform, now normalize using the computed mean and std,
# also add random horizontal and vertical flipping for dataset augmentation

mean = [0.7008, 0.5384, 0.6916]
std = [0.2350, 0.2774, 0.2129]

class Random90Rotation:
    def __call__(self, x):
        return transforms.functional.rotate(x, random.choice([0, 90, 180, 270]))

# add random rotations
train_transform_wRandomRotations = transforms.Compose([
    transforms.ToPILImage(),  # only if your input is not already PIL
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    Random90Rotation(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

In [ ]:
### Assemble the model
mmsen_final = assemble_mmsen_small()
mmsen_final = mmsen_final.to(device, memory_format=torch.channels_last)

In [ ]:
train_loop_regularization(mmsen_final, "mmsen_final", custom_lr=True, lr=1e-4, regularization_method='L2', lambda_=0.005)

In [ ]:
exps = ['mmsen_classifier_10epochs', 'mmsen_final']
labels = ['Baseline', 'Final']
compare_exps(exps, labels)

### Evaluate on validation data

Finally we evaluate the baseline and the final model we have trained on the validation dataset.

In [ ]:
# load the default model
# we call it big now
mmsen_base = assemble_mmsen()
mmsen_base_pth = torch.load('/content/drive/MyDrive/mmsen_classifier_10epochs.pth', weights_only=False, map_location=torch.device(device))
mmsen_base.load_state_dict(mmsen_base_pth['model_state_dict'])
mmsen_base = mmsen_base.to(device, memory_format=torch.channels_last)

In [ ]:
# evaluate
mmsen_base_classes = eval(mmsen_base)

In [ ]:
confusion_matrix(mmsen_base_classes)

In [ ]:
# load the final model
mmsen_final_pth = torch.load('/content/drive/MyDrive/mmsen_final.pth', weights_only=False, map_location=torch.device(device))
mmsen_final.load_state_dict(mmsen_final_pth['model_state_dict'])
mmsen_final = mmsen_final.to(device, memory_format=torch.channels_last)

In [ ]:
# evaluate
mmsen_final_classes = eval(mmsen_final)

In [ ]:
confusion_matrix(mmsen_final_classes)

We managed to prevent the test error from exploding as before, by applying the regularization techniques. <br>
Still, using the selected set of measures against overfitting, the test error stays constant over the course of training. <br>
On the upside ROC-AUC is increasing over the epochs. As this metric is more telling of the performance of the model in practice, this is a decent result. <br>
Unfortunately other metrics are oscilating during training.

Looking at the confusion matrix we see that false positive slightly increase and true negatives decrease, compared to the default model. <br>
On the brightside true positives rise and false negatives decline.